# LES Smagorinsky — 3D flow past a sphere

Large-eddy simulation of incompressible flow past a sphere using the **Smagorinsky** subgrid model:

$$
\nu_t = (C_s \Delta)^2 |S|,\qquad
|S| = \sqrt{2 S_{ij} S_{ij}},\qquad
\Delta = (\Delta x\,\Delta y\,\Delta z)^{1/3}
$$

Solver: fractional-step (Chorin) projection on a Cartesian grid with an immersed sphere  
(geometry matches `LES_smagorinksy`: box $10\times5\times5$, sphere $R=0.5$ at $(3,2.5,2.5)$).

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from les_smagorinsky_sphere import LESConfig, SmagorinskyLESSphere

print("numpy", np.__version__)

## 1. Configure & run the LES

In [ ]:
# Quick demo settings (increase nx/ny/nz and n_steps for production)
cfg = LESConfig(
    nx=48, ny=24, nz=24,
    n_steps=150,
    dt=3.0e-3,
    save_every=15,
    poisson_iters=50,
    Cs=0.17,
    nu=1.0e-3,
    U_inlet=1.0,
)

solver = SmagorinskyLESSphere(cfg)
Re = cfg.U_inlet * 2 * cfg.sphere_radius / cfg.nu
print(f"Grid {cfg.nx}×{cfg.ny}×{cfg.nz}  Δ={solver.delta:.4f}  Re_D≈{Re:.0f}  fluid cells={solver.fluid.sum()}")

state = solver.run()
print(f"Done. snapshots={len(state.snapshots)}  t_end={state.time:.3f}s")

## 2. Time history (KE, max speed, mean $\nu_t$)

In [ ]:
h = state.history
fig_hist = make_subplots(rows=1, cols=3, subplot_titles=("Kinetic energy", "Max |U|", "Mean νₜ"))
fig_hist.add_trace(go.Scatter(x=h["time"], y=h["ke"], mode="lines", name="KE"), row=1, col=1)
fig_hist.add_trace(go.Scatter(x=h["time"], y=h["max_speed"], mode="lines", name="|U|max"), row=1, col=2)
fig_hist.add_trace(go.Scatter(x=h["time"], y=h["mean_nu_t"], mode="lines", name="νt"), row=1, col=3)
fig_hist.update_xaxes(title_text="t [s]")
fig_hist.update_layout(height=360, width=1000, title_text="LES diagnostics", showlegend=False)
fig_hist.show()

## 3. Mid-plane fields (z = mid) — speed, pressure, eddy viscosity

In [ ]:
snap = state.snapshots[-1]
x, y = solver.midplane_coords()
cx, cy, _ = cfg.sphere_center
R = cfg.sphere_radius
phi = np.linspace(0, 2 * np.pi, 200)

# Mask solid in midplane for cleaner plots
solid_xy = solver.solid[:, :, cfg.nz // 2]

def masked(field):
    out = field.astype(float).copy()
    out[solid_xy] = np.nan
    return out

fig_mp = make_subplots(
    rows=1, cols=3,
    subplot_titles=("|U| mid-plane", "Pressure", "Smagorinsky νₜ"),
    horizontal_spacing=0.08,
)
fig_mp.add_trace(go.Heatmap(x=x, y=y, z=masked(snap["speed_xy"]).T, colorscale="Viridis",
                            colorbar=dict(title="|U|", x=0.30, len=0.7)), row=1, col=1)
fig_mp.add_trace(go.Heatmap(x=x, y=y, z=masked(snap["p_xy"]).T, colorscale="RdBu_r",
                            colorbar=dict(title="p", x=0.64, len=0.7)), row=1, col=2)
fig_mp.add_trace(go.Heatmap(x=x, y=y, z=masked(snap["nu_t_xy"]).T, colorscale="Hot",
                            colorbar=dict(title="νt", x=1.02, len=0.7)), row=1, col=3)

for col in (1, 2, 3):
    fig_mp.add_trace(
        go.Scatter(x=cx + R * np.cos(phi), y=cy + R * np.sin(phi),
                   mode="lines", line=dict(color="white", width=2),
                   showlegend=False, hoverinfo="skip"),
        row=1, col=col,
    )

fig_mp.update_xaxes(title_text="x", scaleanchor="y", scaleratio=1)
fig_mp.update_yaxes(title_text="y")
fig_mp.update_layout(height=420, width=1100,
                     title_text=f"Mid-plane at t = {snap['time']:.3f} s")
fig_mp.show()

## 4. 3D visualization — sphere + velocity cones / speed cloud

In [ ]:
xs, ys, zs = solver.sphere_surface_mesh()
stride = snap["stride"]
xg = solver.x[::stride]
yg = solver.y[::stride]
zg = solver.z[::stride]
Xg, Yg, Zg = np.meshgrid(xg, yg, zg, indexing="ij")

# Subsample cones for clarity
step = 2
u3, v3, w3, s3 = snap["u3"], snap["v3"], snap["w3"], snap["speed3"]
mask = s3 > 0.15 * cfg.U_inlet
# Avoid drawing inside sphere
rr = np.sqrt((Xg - cx) ** 2 + (Yg - cy) ** 2 + (Zg - cfg.sphere_center[2]) ** 2)
mask &= rr > R * 1.05

fig3d = go.Figure()
fig3d.add_trace(go.Surface(
    x=xs, y=ys, z=zs,
    colorscale=[[0, "#64748b"], [1, "#94a3b8"]],
    showscale=False, opacity=1.0, name="sphere",
))
fig3d.add_trace(go.Cone(
    x=Xg[mask][::step], y=Yg[mask][::step], z=Zg[mask][::step],
    u=u3[mask][::step], v=v3[mask][::step], w=w3[mask][::step],
    colorscale="Viridis",
    sizemode="absolute",
    sizeref=0.35,
    anchor="tail",
    colorbar=dict(title="|U|", len=0.6),
    name="velocity",
))
fig3d.update_layout(
    title=f"LES Smagorinsky — 3D velocity field (t={snap['time']:.3f}s)",
    scene=dict(
        xaxis_title="x", yaxis_title="y", zaxis_title="z",
        aspectmode="data",
        camera=dict(eye=dict(x=1.4, y=-1.6, z=0.9)),
    ),
    width=900, height=650,
    margin=dict(l=0, r=0, t=50, b=0),
)
fig3d.show()
fig3d.write_html("les_smagorinsky_3d.html", include_plotlyjs="cdn")
print("Saved → les_smagorinsky_3d.html")

## 5. Animate mid-plane speed over time

In [ ]:
frames = []
z0 = masked(state.snapshots[0]["speed_xy"]).T
for s in state.snapshots:
    frames.append(go.Frame(
        data=[go.Heatmap(z=masked(s["speed_xy"]).T, x=x, y=y,
                         colorscale="Viridis", zmin=0, zmax=1.6)],
        name=f"t={s['time']:.3f}",
    ))

fig_anim = go.Figure(
    data=[go.Heatmap(z=z0, x=x, y=y, colorscale="Viridis", zmin=0, zmax=1.6,
                     colorbar=dict(title="|U|"))],
    frames=frames,
)
fig_anim.add_trace(go.Scatter(
    x=cx + R * np.cos(phi), y=cy + R * np.sin(phi),
    mode="lines", line=dict(color="white", width=2), hoverinfo="skip", showlegend=False,
))
fig_anim.update_layout(
    title="LES mid-plane |U| animation",
    xaxis=dict(title="x", scaleanchor="y", scaleratio=1, range=[0, cfg.Lx]),
    yaxis=dict(title="y", range=[0, cfg.Ly]),
    width=800, height=420,
    updatemenus=[{
        "type": "buttons", "showactive": False, "y": 0, "x": 0.05,
        "buttons": [
            {"label": "Play", "method": "animate",
             "args": [None, {"frame": {"duration": 120, "redraw": True},
                             "fromcurrent": True}]},
            {"label": "Pause", "method": "animate",
             "args": [[None], {"frame": {"duration": 0, "redraw": False},
                               "mode": "immediate"}]},
        ],
    }],
    sliders=[{
        "steps": [{"args": [[fr.name], {"frame": {"duration": 0, "redraw": True},
                                        "mode": "immediate"}],
                   "label": fr.name, "method": "animate"} for fr in frames],
        "x": 0.1, "len": 0.85,
    }],
)
fig_anim.show()
fig_anim.write_html("les_smagorinsky_animation.html", include_plotlyjs="cdn")
print("Saved → les_smagorinsky_animation.html")

## 6. Optional: longer / finer run

Uncomment to run a higher-resolution case (slower).

In [ ]:
# cfg_hi = LESConfig(nx=64, ny=32, nz=32, n_steps=400, dt=2.5e-3, save_every=20)
# solver_hi = SmagorinskyLESSphere(cfg_hi)
# state_hi = solver_hi.run()
print("Skipped fine run (uncomment above to enable).")